[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH04/04_04_begin/notebooks/04_04_format_chat_templates.ipynb)

# 04_04 Format data with chat templates

Turn Ridgeline tickets into chat-format training examples, and see exactly what the model reads during training.

These steps don't need a GPU. The default CPU runtime works fine.

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH04")

import ridgeline

In [ ]:
raw = ridgeline.load_tickets(ridgeline.DATA / "synthetic_raw.jsonl")
seeds = ridgeline.load_tickets()
print(f"{len(raw)} synthetic tickets, {len(seeds)} hand-written seed tickets")

## Look at one raw ticket

This is what the generator produced: the ticket, what the support system knows, the labels, and the ideal reply.

In [ ]:
# TODO: Print the first raw ticket as formatted JSON.


## Convert it to a training example

A training example is a short conversation: a system prompt, the customer's ticket, and the ideal assistant reply. The system prompt leaves out the voice rules. The model learns the voice from the replies.

In [ ]:
# TODO: Convert raw[0] with ridgeline.to_training_example, save it as example, and print each message's role and content.


## See it through the chat template

Every model expects conversations in its own exact format. `apply_chat_template` produces that format, and it's what the model reads during training.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-4B-Instruct-2507")

In [ ]:
# TODO: Render the example with tokenizer.apply_chat_template(..., tokenize=False) and print it.


Look inside the assistant turn. Qwen3's template adds an empty `<think></think>` block, left over from its reasoning mode. This model doesn't reason out loud, and you don't want it learning to start every reply with empty tags. `ridgeline.chat_text` applies the template and removes that block. You'll use it from here on, in training and when you run the model.

In [ ]:
print(ridgeline.chat_text(tokenizer, example["messages"]))

Notice the `<|im_start|>` and `<|im_end|>` markers around each turn. If you train with one format and run the model with another, it gets confused. That's why you always use the tokenizer's own template, in training and when you run the model.

## Format every ticket and measure the lengths

Skip any ticket that's missing a field for now. You'll do a full quality check in the next video.

In [ ]:
# TODO: Keep tickets with every required field, format them, and print the shortest, average, and longest token counts.
# Count tokens with ridgeline.count_tokens(tokenizer, example["messages"]).


Every example has to fit inside the model's maximum sequence length, which you set to 2,048 tokens. Next, you'll validate these examples and split them into training and evaluation sets.